**Setup and Installation**

In [1]:
!pip install xgboost -q


In [22]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier
import joblib

sns.set_style('whitegrid')
%matplotlib inline

**Loading the Dataset and Preprocessing**

In [4]:
df = pd.read_csv('final Dataset.csv')
print(df.head())
print(df.shape)


                                  URL  URLLength                      Domain  \
0    https://www.southbankmosaics.com         31    www.southbankmosaics.com   
1            https://www.uni-mainz.de         23            www.uni-mainz.de   
2      https://www.voicefmradio.co.uk         29      www.voicefmradio.co.uk   
3         https://www.sfnmjournal.com         26         www.sfnmjournal.com   
4  https://www.rewildingargentina.org         33  www.rewildingargentina.org   

   DomainLength  IsDomainIP  TLD  URLSimilarityIndex  CharContinuationRate  \
0            24           0  com               100.0              1.000000   
1            16           0   de               100.0              0.666667   
2            22           0   uk               100.0              0.866667   
3            19           0  com               100.0              1.000000   
4            26           0  org               100.0              1.000000   

   TLDLegitimateProb  URLCharProb  ...  Pay  Crypt

Preprocessing


In [46]:
X = df.drop(columns=['label', 'URL'])
y = df['label']   # 1 = legitimate, 0 = phishing

# Drop any remaining non-numeric columns
non_numeric_cols = X.select_dtypes(include=['object']).columns

#These are irrevelant features as these need webpage access to determine which takes time
columns_to_remove = [
    'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title',  'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots',  'IsResponsive',  'HasDescription', 'NoOfPopup','NoOfiFrame', 'HasExternalFormSubmit',
    'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS',  'NoOfSelfRef',  'NoOfEmptyRef',    'NoOfExternalRef','URLSimilarityIndex'
]

if len(non_numeric_cols) > 0:
    print(f"Dropping non-numeric: {list(non_numeric_cols)}")
    X = X.drop(columns=non_numeric_cols)

X = X.drop(columns=[col for col in columns_to_remove if col in X.columns])

# Fill missing values with median
X = X.fillna(X.median())

print("Features shape:", X.shape)
print("Label distribution:\n", y.value_counts())

Dropping non-numeric: ['Domain', 'TLD', 'Title']
Features shape: (235795, 23)
Label distribution:
 label
1    134850
0    100945
Name: count, dtype: int64


**Train-Test Split and Scaling**


In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaler for Logistic Regression only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

Train: 188636 samples
Test:  47159 samples


**Model Training and Selection**

In [48]:
# Logistic Regression with L2 regularization
lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)

# Random Forest with restrictions
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_split=20,
    min_samples_leaf=10, random_state=42, n_jobs=-1
)

# XGBoost with early stopping and regularization
xgb_clf = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    use_label_encoder=False, eval_metric='logloss', random_state=42
)

models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb_clf}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"{name:20} | Test Acc: {acc:.4f} | F1: {f1:.4f}")

Logistic Regression  | Test Acc: 0.9948 | F1: 0.9955
Random Forest        | Test Acc: 0.9951 | F1: 0.9957


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:59:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost              | Test Acc: 0.9950 | F1: 0.9957


In [49]:

estimators = [('lr', lr), ('rf', rf), ('xgb', xgb_clf)]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=0.1),
    cv=5,   # cross-validation inside stacking
    stack_method='predict_proba'
)

stack.fit(X_train_scaled, y_train)
y_pred_stack = stack.predict(X_test_scaled)

print("\n===== Stacking Ensemble =====")
print(f"Accuracy: {accuracy_score(y_test, y_pred_stack):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_stack):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_stack):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_stack):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_stack))


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:00:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:01:48] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:01:50] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:01:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


===== Stacking Ensemble =====
Accuracy: 0.9956
Precision: 0.9942
Recall: 0.9981
F1-score: 0.9961

Confusion Matrix:
[[20031   158]
 [   51 26919]]


Cross validate for checking overfitting

In [54]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(stack, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"5-fold CV accuracy on training set: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")


test_acc = accuracy_score(y_test, y_pred_stack)
print(f"Test accuracy: {test_acc:.4f}")
if test_acc < cv_scores.mean() - 0.02:
    print("⚠️ Possible overfitting – test accuracy lower than CV mean.")
else:
    print("✅ No severe overfitting detected.")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:06:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:07:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:07:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:07:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

5-fold CV accuracy on training set: 0.9958 (+/- 0.0001)
Test accuracy: 0.9956
✅ No severe overfitting detected.


**Evaluation Metrics**

In [50]:
y_proba = stack.predict_proba(X_test_scaled)[:, 1]

In [51]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
import time

# Assuming y_test (true labels) and y_pred (predictions), and y_proba (probabilities)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
specificity = confusion_matrix(y_test, y_pred)[0,0] / (confusion_matrix(y_test, y_pred)[0,0] + confusion_matrix(y_test, y_pred)[0,1])
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1: {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"ROC‑AUC: {auc:.4f}")


# Inference time
start = time.time()
for _ in range(1000):
    _ = stack.predict(X_test_scaled[:1])
avg_time = (time.time() - start) / 1000 * 1000  # ms
print(f"Avg inference time: {avg_time:.2f} ms")

Accuracy: 0.9950
Precision: 0.9932
Recall: 0.9981
F1: 0.9957
Specificity: 0.9908
ROC‑AUC: 0.9986
Avg inference time: 30.71 ms


**Model Saving**

In [52]:
# Save model and scaler
joblib.dump(stack, 'phishguard_model_final.pkl')
joblib.dump(scaler, 'scaler_final.pkl')
joblib.dump(X.columns.tolist(), 'feature_columns_final.pkl')

from google.colab import files
files.download('phishguard_model_final.pkl')
files.download('scaler_final.pkl')
files.download('feature_columns_final.pkl')
print("Model and scaler downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model and scaler downloaded.


In [53]:
X_train.columns


Index(['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate',
       'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain',
       'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
       'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL',
       'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS', 'NoOfURLRedirect',
       'NoOfSelfRedirect'],
      dtype='object')